In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/"

train_df = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_train.csv")
val_df   = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_val.csv")
test_df  = pd.read_csv(f"{BASE_DIR}/weighted_fusion_results/fusion_test.csv")

features = ["DR_prob", "Gl_prob", "AMD_prob", "DED_prob",
            "is_fundus", "is_oct", "is_slitlamp"]

targets = ["DR", "Glaucoma", "AMD", "DED"]

X_train = train_df[features].values
y_train = train_df[targets].values

X_val = val_df[features].values
y_val = val_df[targets].values

X_test = test_df[features].values
y_test = test_df[targets].values

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [ ]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
train_loader = DataLoader(FusionDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader   = DataLoader(FusionDataset(X_val, y_val), batch_size=32, shuffle=False)
test_loader  = DataLoader(FusionDataset(X_test, y_test), batch_size=32, shuffle=False)

In [ ]:
class ResidualFusion(nn.Module):
    def __init__(self, input_dim=7):
        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, input_dim),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        attn = self.attention(x)

        weighted = x * (1 + 0.8 * attn)

        out = self.classifier(weighted)
        return out

In [ ]:
model = ResidualFusion().to(device)

pos_weights = torch.tensor([2.8, 1.5, 1.0, 1.0]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

In [ ]:
SAVE_DIR = os.path.join(BASE_DIR, "residual_fusion_results_final")
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
epochs = 25
best_val_f1 = 0

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # ===== VALIDATION =====
    model.eval()
    val_preds, val_targets = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)

            out = model(xb)
            probs = torch.sigmoid(out)

            val_preds.append(probs.cpu().numpy())
            val_targets.append(yb.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)

    val_bin = (val_preds > 0.5).astype(int)
    val_f1 = f1_score(val_targets, val_bin, average="macro")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "residual_fusion_model.pth"))

    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Val F1: {val_f1:.4f}")

Epoch 1/25 | Loss: 190.4621 | Val F1: 0.8570
Epoch 2/25 | Loss: 73.2715 | Val F1: 0.8633
Epoch 3/25 | Loss: 68.1849 | Val F1: 0.8656
Epoch 4/25 | Loss: 66.1178 | Val F1: 0.8702
Epoch 5/25 | Loss: 65.2448 | Val F1: 0.8702
Epoch 6/25 | Loss: 64.0784 | Val F1: 0.8794
Epoch 7/25 | Loss: 62.9427 | Val F1: 0.8736
Epoch 8/25 | Loss: 62.1161 | Val F1: 0.8702
Epoch 9/25 | Loss: 61.9900 | Val F1: 0.8754
Epoch 10/25 | Loss: 60.8164 | Val F1: 0.8777
Epoch 11/25 | Loss: 60.2666 | Val F1: 0.8858
Epoch 12/25 | Loss: 59.5283 | Val F1: 0.8811
Epoch 13/25 | Loss: 59.5242 | Val F1: 0.8888
Epoch 14/25 | Loss: 58.7476 | Val F1: 0.8813
Epoch 15/25 | Loss: 58.0970 | Val F1: 0.8833
Epoch 16/25 | Loss: 57.8624 | Val F1: 0.8859
Epoch 17/25 | Loss: 57.7507 | Val F1: 0.8864
Epoch 18/25 | Loss: 57.1120 | Val F1: 0.8880
Epoch 19/25 | Loss: 56.9640 | Val F1: 0.8873
Epoch 20/25 | Loss: 56.3233 | Val F1: 0.8878
Epoch 21/25 | Loss: 56.2244 | Val F1: 0.8843
Epoch 22/25 | Loss: 55.8485 | Val F1: 0.8911
Epoch 23/25 | Loss

In [ ]:
model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "residual_fusion_model.pth")))
model.eval()

ResidualFusion(
  (attention): Sequential(
    (0): Linear(in_features=7, out_features=16, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=16, out_features=7, bias=True)
    (4): Sigmoid()
  )
  (classifier): Sequential(
    (0): Linear(in_features=7, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Linear(in_features=16, out_features=4, bias=True)
  )
)

In [ ]:
thresholds = []

for i in range(4):
    best_t, best_f1 = 0.5, 0

    for t in np.linspace(0.65, 0.75, 200):
        preds_bin = (val_preds[:, i] > t).astype(int)
        f1 = f1_score(val_targets[:, i], preds_bin)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print("Optimal thresholds:", thresholds)

Optimal thresholds: [np.float64(0.7444723618090452), np.float64(0.6565326633165829), np.float64(0.7123115577889447), np.float64(0.65)]


In [ ]:
test_preds, test_targets = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        out = model(xb)
        probs = torch.sigmoid(out)

        test_preds.append(probs.cpu().numpy())
        test_targets.append(yb.cpu().numpy())

test_preds = np.vstack(test_preds)
test_targets = np.vstack(test_targets)

In [ ]:
final_preds = np.zeros_like(test_preds)

for i in range(4):
    final_preds[:, i] = (test_preds[:, i] > thresholds[i]).astype(int)

In [ ]:
diseases = ["DR", "Glaucoma", "AMD", "DED"]

results = []

for i, disease in enumerate(diseases):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], final_preds[:, i])
    prec = precision_score(test_targets[:, i], final_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], final_preds[:, i])
    f1 = f1_score(test_targets[:, i], final_preds[:, i])

    results.append([disease, auc, acc, prec, rec, f1])

results_df = pd.DataFrame(results, columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"])
print(results_df)

    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.990655  0.968625   0.690608  0.896057  0.780031
1  Glaucoma  0.975868  0.936360   0.905970  0.731325  0.809333
2       AMD  0.999373  0.995327   0.995939  0.982966  0.989410
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
results_df.to_csv(os.path.join(SAVE_DIR, "metrics.csv"), index=False)
np.save(os.path.join(SAVE_DIR, "thresholds.npy"), thresholds)

torch.save(model.state_dict(), os.path.join(SAVE_DIR, "residual_fusion_model.pth"))

In [ ]:
# ===== MODEL-SPECIFIC THRESHOLD TUNING =====

thresholds = []

threshold_ranges = [
    np.linspace(0.6, 0.75, 200),   # DR
    np.linspace(0.55, 0.7, 200),   # Glaucoma
    np.linspace(0.1, 0.3, 100),    # AMD
    np.linspace(0.4, 0.6, 100)     # DED
]

for i in range(4):
    best_t, best_f1 = 0.5, 0

    for t in threshold_ranges[i]:
        preds_bin = (val_preds[:, i] > t).astype(int)
        f1 = f1_score(val_targets[:, i], preds_bin)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    thresholds.append(best_t)

print("Optimized thresholds:", thresholds)

Optimized thresholds: [np.float64(0.7447236180904523), np.float64(0.55), np.float64(0.16262626262626262), np.float64(0.4)]


In [ ]:
# ===== FIXED THRESHOLD EVALUATION =====

fixed_preds = (test_preds > 0.5).astype(int)

results_fixed = []

for i, disease in enumerate(["DR", "Glaucoma", "AMD", "DED"]):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], fixed_preds[:, i])
    prec = precision_score(test_targets[:, i], fixed_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], fixed_preds[:, i])
    f1 = f1_score(test_targets[:, i], fixed_preds[:, i])

    results_fixed.append([disease, auc, acc, prec, rec, f1])

results_fixed_df = pd.DataFrame(results_fixed,
    columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"])

print("=== Fixed Threshold (0.5) ===")
print(results_fixed_df)

=== Fixed Threshold (0.5) ===
    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.990655  0.961504   0.621560  0.971326  0.758042
1  Glaucoma  0.975868  0.936137   0.854902  0.787952  0.820063
2       AMD  0.999373  0.995327   0.993933  0.984970  0.989431
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
# ===== OPTIMIZED THRESHOLD EVALUATION =====

opt_preds = np.zeros_like(test_preds)

for i in range(4):
    opt_preds[:, i] = (test_preds[:, i] > thresholds[i]).astype(int)

results_opt = []

for i, disease in enumerate(["DR", "Glaucoma", "AMD", "DED"]):
    auc = roc_auc_score(test_targets[:, i], test_preds[:, i])
    acc = accuracy_score(test_targets[:, i], opt_preds[:, i])
    prec = precision_score(test_targets[:, i], opt_preds[:, i], zero_division=0)
    rec = recall_score(test_targets[:, i], opt_preds[:, i])
    f1 = f1_score(test_targets[:, i], opt_preds[:, i])

    results_opt.append([disease, auc, acc, prec, rec, f1])

results_opt_df = pd.DataFrame(results_opt,
    columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"])

print("=== Optimized Thresholds ===")
print(results_opt_df)

=== Optimized Thresholds ===
    Disease       AUC  Accuracy  Precision    Recall        F1
0        DR  0.990655  0.968625   0.690608  0.896057  0.780031
1  Glaucoma  0.975868  0.937027   0.871099  0.773494  0.819400
2       AMD  0.999373  0.994882   0.988967  0.987976  0.988471
3       DED  0.999996  0.999555   0.965517  1.000000  0.982456


In [ ]:
results_fixed_df.to_csv(os.path.join(SAVE_DIR, "metrics_fixed.csv"), index=False)
results_opt_df.to_csv(os.path.join(SAVE_DIR, "metrics_optimized.csv"), index=False)
np.save(os.path.join(SAVE_DIR, "thresholds.npy"), thresholds)